In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

df = pd.read_csv('../data/raw/tender_data.csv')
print(f"Загружено: {df.shape[0]:,} строк × {df.shape[1]} колонок")

Загружено: 4,519 строк × 19 колонок


In [3]:
df['y_success'] = (df['selection_phase'] == 'Завершена').astype(int)

print("Распределение y_success:")
print(df['y_success'].value_counts(normalize=True).round(3))

Распределение y_success:
y_success
0    0.648
1    0.352
Name: proportion, dtype: float64


In [4]:
df['publication_date'] = pd.to_datetime(df['publication_date'])
df = df.sort_values('publication_date').reset_index(drop=True)

print(f"Период: с {df['publication_date'].min()} по {df['publication_date'].max()}")

Период: с 2017-11-22 11:56:21 по 2023-04-24 11:24:08


## Блок 1: Базовая подготовка
- Создана целевая переменная `y_success` из `selection_phase`
- Датасет отсортирован по дате публикации — необходимо для time-based split

In [5]:
df['year'] = df['publication_date'].dt.year
df['month'] = df['publication_date'].dt.month
df['quarter'] = df['publication_date'].dt.quarter
df['day_of_week'] = df['publication_date'].dt.dayofweek  # 0 = понедельник
df['day_of_month'] = df['publication_date'].dt.day
df['is_end_of_quarter'] = df['month'].isin([3, 6, 9, 12]).astype(int)
df['is_end_of_year'] = (df['month'] == 12).astype(int)

# Проверка
print(df[['publication_date', 'year', 'month', 'quarter',
          'day_of_week', 'is_end_of_quarter']].head())

     publication_date  year  month  quarter  day_of_week  is_end_of_quarter
0 2017-11-22 11:56:21  2017     11        4            2                  0
1 2019-02-28 09:19:48  2019      2        1            3                  0
2 2019-07-31 19:39:46  2019      7        3            2                  0
3 2019-09-27 14:14:30  2019      9        3            4                  1
4 2019-09-30 12:56:37  2019      9        3            0                  1


In [6]:
def parse_advance(value):
    """Превращает '30.00%' в 0.30, обрабатывает пропуски и ошибки."""
    if pd.isna(value):
        return 0.0
    try:
        return float(str(value).replace('%', '').strip()) / 100
    except (ValueError, AttributeError):
        return 0.0

df['advance_money_pct'] = df['advance_money'].apply(parse_advance)

print("Распределение advance_money_pct:")
print(df['advance_money_pct'].describe())
print(f"\nПроверка на примерах:")
print(df[['advance_money', 'advance_money_pct']].drop_duplicates().head(10))

Распределение advance_money_pct:
count    4519.000000
mean        0.245357
std         0.207452
min         0.000000
25%         0.100000
50%         0.200000
75%         0.300000
max         1.000000
Name: advance_money_pct, dtype: float64

Проверка на примерах:
   advance_money  advance_money_pct
0            NaN             0.0000
1         50.00%             0.5000
2         10.00%             0.1000
4         30.00%             0.3000
8         15.00%             0.1500
9          5.00%             0.0500
13        12.85%             0.1285
14        80.00%             0.8000
18        20.00%             0.2000
20        18.00%             0.1800


In [7]:
# Логарифм начальной цены — нормализует распределение
df['log_start_price'] = np.log10(df['start_price'].clip(lower=1))

# Обеспечение: отношение к цене (0-1)
df['security_ratio'] = df['tender_security'] / df['start_price']
df['security_ratio'] = df['security_ratio'].fillna(0)

# Флаг: есть обеспечение или нет
df['has_security'] = df['tender_security'].notna().astype(int)

# Признак: есть аванс или нет
df['has_advance'] = (df['advance_money_pct'] > 0).astype(int)

print("Новые финансовые признаки:")
print(df[['log_start_price', 'security_ratio', 'has_security', 'has_advance']].describe())

Новые финансовые признаки:
       log_start_price  security_ratio  has_security  has_advance
count      4519.000000     4519.000000   4519.000000  4519.000000
mean          9.077286        0.030212      0.951981     0.872981
std           0.347390        0.149702      0.213831     0.333031
min           8.698970        0.000000      0.000000     0.000000
25%           8.825009        0.010000      1.000000     1.000000
50%           8.976583        0.025000      1.000000     1.000000
75%           9.219760        0.050000      1.000000     1.000000
max          11.131759       10.000000      1.000000     1.000000


In [8]:
# Логарифм начальной цены — нормализует распределение
df['log_start_price'] = np.log10(df['start_price'].clip(lower=1))

# Обеспечение: отношение к цене (0-1)
df['security_ratio'] = df['tender_security'] / df['start_price']
df['security_ratio'] = df['security_ratio'].fillna(0)

# Флаг: есть обеспечение или нет
df['has_security'] = df['tender_security'].notna().astype(int)

# Признак: есть аванс или нет
df['has_advance'] = (df['advance_money_pct'] > 0).astype(int)

print("Новые финансовые признаки:")
print(df[['log_start_price', 'security_ratio', 'has_security', 'has_advance']].describe())

Новые финансовые признаки:
       log_start_price  security_ratio  has_security  has_advance
count      4519.000000     4519.000000   4519.000000  4519.000000
mean          9.077286        0.030212      0.951981     0.872981
std           0.347390        0.149702      0.213831     0.333031
min           8.698970        0.000000      0.000000     0.000000
25%           8.825009        0.010000      1.000000     1.000000
50%           8.976583        0.025000      1.000000     1.000000
75%           9.219760        0.050000      1.000000     1.000000
max          11.131759       10.000000      1.000000     1.000000


In [9]:
# История ЗАКАЗЧИКА (по ИНН)
# expanding() накапливает статистику от начала до текущей строки
# shift() смещает на 1, чтобы не включать текущую строку в свою же историю

df['customer_total_tenders'] = (
    df.groupby('customer_inn').cumcount()
)

df['customer_success_rate'] = (
    df.groupby('customer_inn')['y_success']
      .expanding().mean()
      .shift(1)  # ВАЖНО: исключаем текущую строку
      .reset_index(level=0, drop=True)
      .fillna(0.5)  # для первого тендера заказчика — нейтральная оценка
)

print("Примеры истории заказчика:")
sample_customer = df[df['customer_inn'].notna()]['customer_inn'].value_counts().index[0]
cols = ['publication_date', 'customer_inn', 'y_success',
        'customer_total_tenders', 'customer_success_rate']
print(df[df['customer_inn'] == sample_customer][cols].head(10))

Примеры истории заказчика:
       publication_date  customer_inn  y_success  customer_total_tenders  \
15  2019-10-09 11:17:41  7.728382e+09          1                     0.0   
126 2019-11-08 13:56:30  7.728382e+09          1                     1.0   
227 2019-12-04 17:06:39  7.728382e+09          1                     2.0   
249 2019-12-12 10:38:53  7.728382e+09          1                     3.0   
279 2019-12-30 11:33:40  7.728382e+09          0                     4.0   
309 2020-02-10 11:47:36  7.728382e+09          1                     5.0   
325 2020-02-13 17:34:55  7.728382e+09          0                     6.0   
338 2020-02-14 14:11:23  7.728382e+09          0                     7.0   
341 2020-02-14 14:21:50  7.728382e+09          0                     8.0   
420 2020-03-26 14:15:44  7.728382e+09          0                     9.0   

     customer_success_rate  
15                1.000000  
126               1.000000  
227               1.000000  
249               1.

In [10]:
df['region_total_tenders'] = (
    df.groupby('customer_region_code').cumcount()
)

df['region_success_rate'] = (
    df.groupby('customer_region_code')['y_success']
      .expanding().mean()
      .shift(1)
      .reset_index(level=0, drop=True)
      .fillna(0.5)
)

# То же для процедуры
df['procedure_success_rate'] = (
    df.groupby('procedure')['y_success']
      .expanding().mean()
      .shift(1)
      .reset_index(level=0, drop=True)
      .fillna(0.5)
)

print("Распределение исторических success_rate:")
print(df[['customer_success_rate', 'region_success_rate',
          'procedure_success_rate']].describe())

Распределение исторических success_rate:
       customer_success_rate  region_success_rate  procedure_success_rate
count            4512.000000          4519.000000             4519.000000
mean                0.321447             0.299113                0.318172
std                 0.339636             0.178563                0.171994
min                 0.000000             0.000000                0.000000
25%                 0.000000             0.172414                0.208032
50%                 0.238095             0.333333                0.316239
75%                 0.540210             0.386223                0.395185
max                 1.000000             1.000000                1.000000


## History-features
Рассчитана история по заказчикам, регионам и процедурам **только на прошлых данных**
(через `expanding().mean().shift(1)`), что предотвращает утечку данных при обучении.
Это самые информативные признаки для задачи прогнозирования.

In [11]:
# Финальный список колонок для модели
FEATURE_COLS = [
    # Числовые
    'start_price', 'log_start_price', 'tender_security',
    'security_ratio', 'has_security',
    'advance_money_pct', 'has_advance',

    # Временные
    'year', 'month', 'quarter', 'day_of_week',
    'day_of_month', 'is_end_of_quarter', 'is_end_of_year',

    # History
    'customer_total_tenders', 'customer_success_rate',
    'region_total_tenders', 'region_success_rate',
    'procedure_success_rate',

    # Категориальные (пойдут в cat_features CatBoost)
    'procedure', 'legislation', 'for_small_business',
    'customer_region_code', 'customer_region',
]

CAT_FEATURES = [
    'procedure', 'legislation', 'for_small_business',
    'customer_region_code', 'customer_region',
]

TARGET = 'y_success'

# Проверка пропусков
print("Пропуски в признаках:")
missing = df[FEATURE_COLS].isnull().sum()
print(missing[missing > 0])

Пропуски в признаках:
tender_security           217
customer_total_tenders      7
customer_success_rate       7
dtype: int64


In [12]:
# CatBoost умеет с пропусками, но для категориальных лучше заполнить
for col in CAT_FEATURES:
    df[col] = df[col].fillna('Unknown').astype(str)

# Числовые — оставляем NaN, CatBoost сам обработает
# Но специально проверим tender_security — там 4.8% пропусков
# Это значит "обеспечение не требуется" → можно заполнить 0
df['tender_security'] = df['tender_security'].fillna(0)

# Итоговая проверка
print("Пропуски после обработки:")
print(df[FEATURE_COLS].isnull().sum().sum(), "всего пропусков")

Пропуски после обработки:
14 всего пропусков


In [13]:
# 80/20 по времени
split_date = df['publication_date'].quantile(0.8)
print(f"Граница train/test: {split_date}")

train_mask = df['publication_date'] < split_date
test_mask = ~train_mask

print(f"\nTrain: {train_mask.sum():,} строк ({df[train_mask]['publication_date'].min()} - {df[train_mask]['publication_date'].max()})")
print(f"Test:  {test_mask.sum():,} строк ({df[test_mask]['publication_date'].min()} - {df[test_mask]['publication_date'].max()})")

print(f"\nБаланс классов train: {df[train_mask][TARGET].mean():.3f}")
print(f"Баланс классов test:  {df[test_mask][TARGET].mean():.3f}")

Граница train/test: 2022-10-13 17:15:10.800000

Train: 3,615 строк (2017-11-22 11:56:21 - 2022-10-13 16:49:52)
Test:  904 строк (2022-10-13 17:53:09 - 2023-04-24 11:24:08)

Баланс классов train: 0.338
Баланс классов test:  0.407


In [14]:
import os
os.makedirs('C:/Users/HP/PycharmProjects/Tender/data/processed', exist_ok=True)

# Сохраняем ВЕСЬ датасет (train+test в одном файле, с признаком is_train)
df_to_save = df[FEATURE_COLS + [TARGET, 'publication_date']].copy()
df_to_save['is_train'] = train_mask.values

df_to_save.to_parquet('C:/Users/HP/PycharmProjects/Tender/data/processed/tender_data_v2.parquet', index=False)
print(f"Сохранено: data/processed/tender_data_v2.parquet")
print(f"Размер: {df_to_save.shape}")
print(f"Колонки: {df_to_save.columns.tolist()}")

Сохранено: data/processed/tender_data_v2.parquet
Размер: (4519, 27)
Колонки: ['start_price', 'log_start_price', 'tender_security', 'security_ratio', 'has_security', 'advance_money_pct', 'has_advance', 'year', 'month', 'quarter', 'day_of_week', 'day_of_month', 'is_end_of_quarter', 'is_end_of_year', 'customer_total_tenders', 'customer_success_rate', 'region_total_tenders', 'region_success_rate', 'procedure_success_rate', 'procedure', 'legislation', 'for_small_business', 'customer_region_code', 'customer_region', 'y_success', 'publication_date', 'is_train']


In [15]:
# Сохраняем списки признаков в Python-файл для переиспользования
features_code = f'''"""
Константы проекта: списки признаков для модели.
Автогенерация из notebooks/03_feature_engineering.ipynb
"""

FEATURE_COLS = {FEATURE_COLS}

CAT_FEATURES = {CAT_FEATURES}

TARGET = '{TARGET}'
'''

with open('../src/features.py', 'w', encoding='utf-8') as f:
    f.write(features_code)

print("Сохранено: src/features.py")

Сохранено: src/features.py
